### Documentation du module de NLP et Tutoriel Jupyter  

classification de films avec le pipeline NLP Naive Bayes / Regression logistique combiner avec le module d'optimisation et sélection de modèles

## Pipeline Complet: NLP + Classification Multilabel

**Étapes:**
1. **Prétraitement du texte (X)**: Tokenization → Suppression stopwords → TF-IDF
2. **Encodage des labels (y)**: Convertir chaque genre en colonne binaire (multilabel binarization)
3. **Entraînement**: Utiliser LogisticRegression avec OneVsRest (approche standard pour multilabel)
4. **Évaluation**: Metrics adaptées au multilabel (Hamming Loss, Precision, Recall, F1)

Dataset link : https://www.kaggle.com/datasets/kishoreramb/movies-dataset
https://www.kaggle.com/datasets/abdallahwagih/spam-emails  
https://www.kaggle.com/datasets/ganiyuolalekan/spam-assassin-email-classification-dataset

In [3]:
import sys 
sys.path.append(r"E:\cours ifri\Programmation et BD\Python\Pdf et tpcours\Concepts et Application\Apprentissage Automatique\ifri_mini_ml_lib")

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ifri_mini_ml_lib.preprocessing.preparation.encoding import OneHotEncoder, CategoricalEncoder
from ifri_mini_ml_lib.preprocessing.text.stop_word import StopWordRemover
from ifri_mini_ml_lib.preprocessing.text.tf_idf import TFIDFVectorizer
from ifri_mini_ml_lib.preprocessing.preparation.tokenization import Tokenizer
from ifri_mini_ml_lib.preprocessing.preparation.splitting import DataSplitter

from ifri_mini_ml_lib.metrics.classification import f1_score, recall, precision, accuracy

# Modeles de Classification
from ifri_mini_ml_lib.classification.logistic_regression import LogisticRegression

# Cross Validation
from ifri_mini_ml_lib.model_selection.cross_validation import k_fold_cross_validation


In [36]:
df = pd.read_csv("spam.csv")


In [7]:
df.isna().sum()

Category    0
Message     0
dtype: int64

No missing values

In [37]:
splitter = DataSplitter(seed=42)


X = df[["Message"]]
y = df["Category"]




X_train, X_test, y_train, y_test = splitter.train_test_split(X, y, test_size=0.2)

X_train, X_test, y_train, y_test = pd.DataFrame(X_train) , pd.DataFrame(X_test) , pd.DataFrame(y_train), pd.DataFrame(y_test)
# Proportions:
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 4458, Test: 1114


In [38]:
# Pipeline NLP simple: Tokenization → Stopwords → TF-IDF

# 1. Tokenizer
tokenizer = Tokenizer()

# 2. Stop words remover

stopwords_remover = StopWordRemover(language='english')

# 3. TF-IDF

tfidf = TFIDFVectorizer(max_features = 1000)



In [39]:
X.describe()

,Message
count,5572
unique,5157
top,"Sorry, I'll call later"
freq,30


In [40]:
X_train_tokens = X_train['Message'].apply(tokenizer.tokenize)
X_train_tokens.head()


4293                                            [g, w, r]
1978    [reply, to, win, 100, weekly, where, will, the...
3989    [hello, sort, of, out, in, town, already, that...
3935    [how, come, guoyang, go, n, tell, her, then, u...
4078    [hey, sathya, till, now, we, dint, meet, not, ...
Name: Message, dtype: object

In [41]:
X_train_without_stop_words = X_train_tokens.apply(stopwords_remover.fit_transform)
X_train_without_stop_words.head()

4293                                            [g, w, r]
1978    [reply, win, 100, weekly, 2006, fifa, world, c...
3989    [hello, sort, town, already, dont, rush, home,...
3935                                [guoyang, n, u, told]
4078    [hey, sathya, till, dint, even, single, time, ...
Name: Message, dtype: object

In [42]:
X_train_encoded = pd.DataFrame(tfidf.fit_transform(X_train_without_stop_words.tolist()))
X_train_encoded.head()

,0,1,2,3,4,5,6,7,8,9,...,990,991,992,993,994,995,996,997,998,999
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
# Observez la structure des genres
print("=== OBSERVATION DES GENRES ===\n")

# Exemples de genres bruts
print("Exemples de genres (premiers 10):")
print(y.head(10))

print(f"\nNombre total de films: {len(y)}")
print(f"Type de données: {type(y.iloc[0])}")

# Tous les genres uniques
all_genres_str = ' '.join(y.values)
unique_genres = set(all_genres_str.split())
print(f"\nGenres uniques totaux: {sorted(unique_genres)}")
print(f"Nombre de genres uniques: {len(unique_genres)}")

# Nombre de genres par film
y_genre_count = y.str.split().apply(len)
print(f"\nNombre de genres par film:")
print(y_genre_count.value_counts().sort_index())

=== OBSERVATION DES GENRES ===

Exemples de genres (premiers 10):
0     ham
1     ham
2    spam
3     ham
4     ham
5    spam
6     ham
7     ham
8    spam
9    spam
Name: Category, dtype: object

Nombre total de films: 5572
Type de données: <class 'str'>

Genres uniques totaux: ['ham', 'spam']
Nombre de genres uniques: 2

Nombre de genres par film:
Category
1    5572
Name: count, dtype: int64


In [44]:
# Faire le prétraitement du test de la meme facon que le train


# Tokenizer -> StopWords -> TFIDFVectorizer


X_test_tokens = X_test['Message'].apply(tokenizer.tokenize)

X_test_without_stop_words = X_test_tokens.apply(stopwords_remover.transform)

X_test_encoded = pd.DataFrame(tfidf.transform(X_test_without_stop_words.tolist()))

print(f"X_test shape: {X_test_encoded.shape}")
print(f"X_test: {len(X_test_encoded)}")


X_test shape: (1114, 1000)
X_test: 1114


In [45]:
# ÉTAPE 2: Encodage  des catégories(y_train, y_test) 

le = CategoricalEncoder('label')

y_train = le.fit_transform(y_train)

In [46]:
y_test = le.transform(y_test)

In [47]:
# ÉTAPE 3: Classification 
X_train = X_train_encoded
y_train_processed = y_train.squeeze() if isinstance(y_train, pd.DataFrame) else np.ravel(y_train)
y_test_processed = y_test.squeeze() if isinstance(y_test, pd.DataFrame) else np.ravel(y_test)

print("ÉTAPE 3: Entraînement du classifieur de spam avec la Regression Logistique")

model = LogisticRegression()  # logistic regression 

score = k_fold_cross_validation(model=model , X=X_train , y=y_train_processed , metric=f1_score , stratified=True)

print(f"Cross-validation - F1_Score global {score[0]}   ± {score[1]}")

# IMPORTANT: Réentraîner le modèle sur toutes les données d'entraînement
print("\nRéentraînement du modèle sur l'ensemble complet...")
model.fit(X_train, y_train_processed)

ÉTAPE 3: Entraînement du classifieur de spam avec la Regression Logistique
Cross-validation - F1_Score global 0.0   ± 0.0

Réentraînement du modèle sur l'ensemble complet...


In [48]:
# Métriques

y_pred = model.predict(X_test_encoded)

print(f"Métriques finales sur l'ensemble de test\n")

# Accuracy (fonctionne pour multiclass)
print(f"Accuracy = {accuracy(y_test_processed , y_pred):.2f}")

# Macro-average: moyenne pour CHAQUE classe
all_classes = np.unique(y_test_processed)
precision_macro = np.mean([precision(y_test_processed, y_pred, positive_class=c) for c in all_classes])
recall_macro = np.mean([recall(y_test_processed, y_pred, positive_class=c) for c in all_classes])
f1_macro = np.mean([f1_score(y_test_processed, y_pred, positive_class=c) for c in all_classes])

print(f"Precision (macro) = {precision_macro:.2f}")
print(f"Recall (macro) = {recall_macro:.2f}")
print(f"F1_Score (macro) = {f1_macro:.2f}")

# Aussi pour la classe positive (supposée être 1)
print(f"\n--- Classe 1 (positive_class) ---")
print(f"Precision (class 1) = {precision(y_test_processed , y_pred, positive_class=1):.2f}")
print(f"Recall (class 1) = {recall(y_test_processed , y_pred, positive_class=1):.2f}")
print(f"F1_Score (class 1) = {f1_score(y_test_processed , y_pred, positive_class=1):.2f}")

Métriques finales sur l'ensemble de test

Accuracy = 0.87
Precision (macro) = 0.43
Recall (macro) = 0.50
F1_Score (macro) = 0.46

--- Classe 1 (positive_class) ---
Precision (class 1) = 0.00
Recall (class 1) = 0.00
F1_Score (class 1) = 0.00


In [49]:

y_pred = model.predict(X_test_encoded)
print(f"Métriques finales sur l'ensemble de test\n")

# Accuracy (fonctionne pour multiclass)
print(f"Accuracy = {accuracy(y_test_processed , y_pred):.2f}")

# Macro-average: moyenne pour CHAQUE classe
all_classes = np.unique(y_test_processed)
precision_macro = np.mean([precision(y_test_processed, y_pred, positive_class=c) for c in all_classes])
recall_macro = np.mean([recall(y_test_processed, y_pred, positive_class=c) for c in all_classes])
f1_macro = np.mean([f1_score(y_test_processed, y_pred, positive_class=c) for c in all_classes])

print(f"Precision (macro) = {precision_macro:.2f}")
print(f"Recall (macro) = {recall_macro:.2f}")
print(f"F1_Score (macro) = {f1_macro:.2f}")

# Aussi pour la classe positive (supposée être 1)
print(f"\n--- Classe 1 (positive_class) ---")
print(f"Precision (class 1) = {precision(y_test_processed , y_pred, positive_class=1):.2f}")
print(f"Recall (class 1) = {recall(y_test_processed , y_pred, positive_class=1):.2f}")
print(f"F1_Score (class 1) = {f1_score(y_test_processed , y_pred, positive_class=1):.2f}")

Métriques finales sur l'ensemble de test

Accuracy = 0.87
Precision (macro) = 0.43
Recall (macro) = 0.50
F1_Score (macro) = 0.46

--- Classe 1 (positive_class) ---
Precision (class 1) = 0.00
Recall (class 1) = 0.00
F1_Score (class 1) = 0.00


In [22]:
y.value_counts()


Category
ham     4825
spam     747
Name: count, dtype: int64

In [50]:
# Changer les poids et attribués un plus grand poids a spam

ham_count = 4825
spam_count = 747

spam_weight = ham_count / spam_count

print(spam_weight)

6.459170013386881


In [51]:
y_test_new = y_test.squeeze().apply(lambda x : 1 if x ==0 else 7)

y_test_new.value_counts()

Category
1    965
7    149
Name: count, dtype: int64

In [52]:
# Réentrainement

y_train_processed = y_train.squeeze() if isinstance(y_train, pd.DataFrame) else np.ravel(y_train)
y_test_processed = y_test_new.squeeze() if isinstance(y_test_new, pd.DataFrame) else np.ravel(y_test_new)

print("ÉTAPE 3: Entraînement du classifieur de spam avec la Regression Logistique")

model = LogisticRegression()  # logistic regression 

score = k_fold_cross_validation(model=model , X=X_train_encoded , y=y_train_processed , metric=f1_score , stratified=True)

print(f"Cross-validation - F1_Score global {score[0]}   ± {score[1]}")

# IMPORTANT: Réentraîner le modèle sur toutes les données d'entraînement
print("\nRéentraînement du modèle sur l'ensemble complet...")
model.fit(X_train, y_train_processed)

ÉTAPE 3: Entraînement du classifieur de spam avec la Regression Logistique
Cross-validation - F1_Score global 0.0   ± 0.0

Réentraînement du modèle sur l'ensemble complet...


In [53]:
probas = model.predict_proba(X_test_encoded)

preds = (probas >= 0.2).astype(np.int8)

precision_macro = np.mean([precision(y_test_processed, preds, positive_class=c) for c in all_classes])
recall_macro = np.mean([recall(y_test_processed, preds, positive_class=c) for c in all_classes])
f1_macro = np.mean([f1_score(y_test_processed, preds, positive_class=c) for c in all_classes])

print(f"Precision (macro) = {precision_macro:.2f}")
print(f"Recall (macro) = {recall_macro:.2f}")
print(f"F1_Score (macro) = {f1_macro:.2f}")

# Aussi pour la classe positive (supposée être 1)
print(f"\n--- Classe 1 (positive_class) ---")
print(f"Precision (class 1) = {precision(y_test_processed , preds, positive_class=1):.2f}")
print(f"Recall (class 1) = {recall(y_test_processed , preds, positive_class=1):.2f}")
print(f"F1_Score (class 1) = {f1_score(y_test_processed , preds, positive_class=1):.2f}")


Precision (macro) = 0.00
Recall (macro) = 0.00
F1_Score (macro) = 0.00

--- Classe 1 (positive_class) ---
Precision (class 1) = 0.00
Recall (class 1) = 0.00
F1_Score (class 1) = 0.00
